# Lumen final batch: the sealed test, then the remaining zero-shot baselines

**This notebook runs the one-time evaluation on VizWiz test.** The runner claims the seal on the Hub before loading any test answer; once claimed it can never run again, even after a crash. Download `lumen-results` when it finishes and commit `results/test/FINAL_TEST_RUN.json` immediately.

Then LFM2.5-VL-450M, SmolVLM2-256M and SmolVLM2-2.2B zero-shot (prompt chosen on dev_select, then val), as for SmolVLM2-500M in Stage 2. Rerunnable: a baseline with a committed val result is skipped; the test stage refuses to run twice.

In [ ]:
# Batch session (pushed by the Kaggle API). Only /kaggle/working/lumen-results is saved as output:
# the clone, VizWiz data, prediction files, and checkpoints all live in /kaggle/temp, which isn't.
import os
import time

T0 = time.time()
GPU_HOURS_LEFT_AT_START = 30.0  # <- set from `kaggle quota`
SESSION_HOURS = 11.0  # Kaggle stops a session at 12 h; training stops cleanly before this
TEMP = "/kaggle/temp"
os.makedirs(TEMP, exist_ok=True)
os.environ.update(
    LUMEN_KAGGLE_WORKDIR=TEMP,
    LUMEN_TEMP_ROOT=TEMP,
    LUMEN_DATA_DIR=f"{TEMP}/lumen-data",
    LUMEN_PREDICTIONS_DIR=f"{TEMP}/lumen-predictions",
    LUMEN_CHECKPOINT_DIR=f"{TEMP}/lumen-checkpoints",
    LUMEN_OUTPUT_DIR="/kaggle/working/lumen-results",
    LUMEN_DEADLINE_UNIX=str(T0 + SESSION_HOURS * 3600),
)

In [ ]:
# Clone or update the private repo. The GitHub token never appears in a command line, a URL,
# notebook output, an exception, or on disk: git receives it only as an HTTP auth header through
# environment variables set for the git process alone (tests/test_notebook_clone.py enforces this).
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/JCHETAN26/lumen.git"
WORKDIR = os.environ.get("LUMEN_KAGGLE_WORKDIR", "/kaggle/working")


def _git(*args: str) -> None:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_TRACE")}
    env.update(
        GIT_TERMINAL_PROMPT="0",  # fail instead of prompting for credentials
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True, cwd=WORKDIR)
    message = (result.stderr or result.stdout or "").strip()
    for secret in (token, basic):  # redact, in case git ever echoes the header
        message = message.replace(secret, "***")
    del token, basic, env
    if result.returncode != 0:
        raise RuntimeError(
            f"git {args[0]} failed (exit {result.returncode}): {message[-800:]}"
        ) from None


os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(WORKDIR, "lumen")):
    _git("clone", "--depth", "1", REPO_URL, "lumen")
else:
    _git("-C", "lumen", "pull", "--ff-only", REPO_URL, "main")
head = subprocess.run(
    ["git", "-C", os.path.join(WORKDIR, "lumen"), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print("repo at", head)

In [ ]:
# Helpers: each runner stage as its own process, sharing the session's deadline and quota.
REPO = f"{TEMP}/lumen"
LOG = []


def hours_left_in_session() -> float:
    return SESSION_HOURS - (time.time() - T0) / 3600


def stage(*args: str, gpu: bool = True) -> int:
    """Run one runner stage with the quota left this week (measured from the session start)."""
    cmd = ["uv", "run", "python", "kaggle/run_stages.py", *args]
    if gpu:
        left = GPU_HOURS_LEFT_AT_START - (time.time() - T0) / 3600
        cmd += ["--gpu-hours-left", f"{left:.2f}"]
    line = " ".join(args)
    print(f"\n######## {line}  (session {(time.time() - T0) / 3600:.2f} h)", flush=True)
    shell = "source /kaggle/working/lumen-env.sh && " + " ".join(cmd)
    code = subprocess.run(["bash", "-c", shell], cwd=REPO).returncode
    LOG.append((line, code, round((time.time() - T0) / 3600, 2)))
    print(f"######## exit {code}", flush=True)
    return code

In [ ]:
# Setup (uv environment in /kaggle/temp) and the machine record.
subprocess.run(["bash", "kaggle/setup.sh"], cwd=REPO, check=True)
stage("env", gpu=False)

In [ ]:
# VizWiz files (pinned sha256), splits checked against the committed ones. ~15 GB into /kaggle/temp.
assert stage("data", gpu=False) == 0

In [ ]:
# HF token for the private Hub repos: exported to the stages, never printed.
import kaggle_secrets

os.environ["HF_TOKEN"] = kaggle_secrets.UserSecretsClient().get_secret("HF_TOKEN")
assert stage("storage_check", gpu=False) == 0

In [ ]:
# The sealed final test: the dev-chosen fine-tune (results/finetune/choice.json) and the matched
# zero-shot SmolVLM2-500M on all 8,000 test questions, once. The plan line first (claims nothing).
from pathlib import Path

stage("final_test", "--dry-run", gpu=False)
if (Path(REPO) / "results" / "test" / "FINAL_TEST_RUN.json").exists():
    print("final test already ran (record committed): skipped")
else:
    stage("final_test", "--final-test-evaluation")

In [ ]:
# Zero-shot baselines, as in Stage 2: prompt chosen on the 512-question dev_select subset, then val
# with that prompt at the default resolution. Baseline runs resume only from this session's disk and
# don't watch the session clock, so a run starts only if its prior estimate (from the runner's
# dry runs: dev_select = both prompts on 512 questions, val ~ 8.4x one dev_select prompt) fits in
# the time left. A skipped run happens in the next session; finished steps are skipped by result.
GPU_H = {  # model: (dev_select, val), prior GPU hours
    "lfm2.5-vl-450m": (0.46, 1.95),
    "smolvlm2-256m": (0.38, 1.60),
    "smolvlm2-2.2b": (1.28, 5.40),
}


def has(model: str, split: str) -> bool:
    return any((Path(REPO) / "results" / "baseline" / model).glob(f"{split}-*.json")) or any(
        Path("/kaggle/working/lumen-results/baseline", model).glob(f"{split}-*.json")
    )


for model, (h_dev, h_val) in GPU_H.items():
    if not has(model, "dev_select"):
        if hours_left_in_session() < h_dev + 0.25:
            print(f"skip {model} dev_select: {hours_left_in_session():.2f} h left in the session")
            continue
        if stage("baseline", "--model", model, "--split", "dev_select", "--prompt", "both") != 0:
            continue
    if not has(model, "val"):
        if hours_left_in_session() < h_val + 0.25:
            print(f"skip {model} val: {hours_left_in_session():.2f} h left in the session")
            continue
        stage("baseline", "--model", model, "--split", "val", "--prompt", "dev-selected")

stage("baseline_summary", gpu=False)

In [ ]:
# What ran, and what is in the saved output (aggregate results only).
for line, code, h in LOG:
    print(f"{h:6.2f} h  exit {code}  {line}")
subprocess.run(["find", "/kaggle/working", "-maxdepth", "3"])